In [60]:
from dotenv import load_dotenv

_ = load_dotenv()

In [61]:
!pip install langchain_community
!pip install langchain_openai
!pip install langchain-groq #freemodel
!pip install langgraph.checkpoint.sqlite

In [62]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
import os
from google.colab import userdata

In [63]:
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
tool = TavilySearchResults(max_results=2)

In [64]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# memory = SqliteSaver.from_conn_string(':memory:')
memory = SqliteSaver(sqlite3.connect(':memory:', check_same_thread=False))

In [65]:
from uuid import uuid4
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage

"""
In previous examples we've annotated the `messages` state key
with the default `operator.add` or `+` reducer, which always
appends new messages to the end of the existing messages array.

Now, to support replacing existing messages, we annotate the
`messages` key with a customer reducer function, which replaces
messages with the same `id`, and appends them otherwise.
"""
def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # assign ids to messages that don't have them
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    # merge the new messages with the existing messages
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            # replace any existing messages with the same id
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            # append any new messages to the end
            merged.append(message)
    return merged

# we replaced the operator.add annotation with the replace annotation.
# For any new message that is generated, we will generate an id for it and append iff there is no existing message with the same id
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

In [66]:
class Agent:
    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"] # we instructed the model to interrepu before calling the action node, so human can interact
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        print(state)
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [67]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatGroq(temperature=0, model_name="llama-3.1-8b-instant")
zarvis_2_o = Agent(model, [tool], system=prompt, checkpointer=memory)

In [68]:
messages = [HumanMessage(content="Whats the weather in Haldwani?")]
thread = {"configurable": {"thread_id": "1"}}
for event in zarvis_2_o.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [HumanMessage(content='Whats the weather in Haldwani?', additional_kwargs={}, response_metadata={}, id='998cb8ba-e952-4f17-acbb-e87ca57c63fd'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '02f4pq1vf', 'function': {'arguments': '{"query":"Haldwani weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.034000551, 'completion_tokens_details': None, 'prompt_time': 0.038102155, 'prompt_tokens_details': None, 'queue_time': 0.005504163, 'total_time': 0.072102706}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-9271-7921-a11b-75158259e471-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Haldwani weather'}, 'id': '02f4pq1vf', 'type': 'tool_call'

⏫ above the langraph interrupted before calling the acton node for executing the tavily searchaaasa

In [69]:
zarvis_2_o.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in Haldwani?', additional_kwargs={}, response_metadata={}, id='998cb8ba-e952-4f17-acbb-e87ca57c63fd'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '02f4pq1vf', 'function': {'arguments': '{"query":"Haldwani weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.034000551, 'completion_tokens_details': None, 'prompt_time': 0.038102155, 'prompt_tokens_details': None, 'queue_time': 0.005504163, 'total_time': 0.072102706}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-9271-7921-a11b-75158259e471-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Haldwani weather'}, 'id': '02f4pq1vf'



```
StateSnapshot:
  created_at: 2026-03-12T08:10:07.156078+00:00

  values:
    messages:

      - type: HumanMessage
        id: 273c040a-4e92-48d4-94ec-adcad27bbfbb
        content: "Whats the weather in Haldwani?"

      - type: AIMessage
        id: lc_run--019ce118-86f1-7522-8dac-6e7955addf6a-0
        content: ""

        tool_calls:
          - id: jrnq1f6yf
            name: tavily_search_results_json
            args:
              query: "Haldwani weather"

        model_info:
          model_name: llama-3.1-8b-instant
          model_provider: groq
          service_tier: on_demand
          system_fingerprint: fp_d317489708
          finish_reason: tool_calls

        token_usage:
          prompt_tokens: 352
          completion_tokens: 22
          total_tokens: 374

        timing:
          prompt_time: 0.024028489
          completion_time: 0.034694279
          queue_time: 0.005281166
          total_time: 0.058722768

  next_step:
    - action

  config:
    thread_id: "1"
    checkpoint_ns: ""
    checkpoint_id: 1f11deae-1b62-6b46-8001-9c21f40c72f0

  parent_config:
    thread_id: "1"
    checkpoint_ns: ""
    checkpoint_id: 1f11deae-1981-6e3d-8000-b766064fd221

  metadata:
    source: loop
    step: 1
    parents: {}

  tasks:
    - id: bbf7588d-3f6f-8929-53e6-761ebc4263df
      name: action
      path:
        - __pregel_pull
        - action
      error: null
      interrupts: []
      state: null
      result: null

  interrupts: []
```



In [70]:
zarvis_2_o.graph.get_state(thread).next

('action',)

In [71]:
# continue after interrupt
for event in zarvis_2_o.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'Haldwani weather'}, 'id': '02f4pq1vf', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Weather Haldwani in March 2026: Temperature & Climate\', \'url\': \'https://en.climate-data.org/asia/india/uttarakhand/haldwani-4074/t/march-3/\', \'content\': \'In Haldwani, March brings rainfall levels ranging from 0.36/0.01 mm/in to 2.64/0.10 mm/in.\\n13.03 is the rainiest day with 2.64/0.10 mm/in, while 25.03 is the driest with 0.36/0.01 mm/in.\\nDuring the first part of the month, rainfall averages 1.00/0.04 mm/in, with 0 wet days.\\nThe middle third sees 1.80/0.07 mm/in, with 0 rainy days.\\nThe final third averages 0.80/0.03 mm/in, with 0 rainy days.\\nThe 10-day peak in rainfall occurs between 11-20 (1.80/0.07 mm/in).\\nThe driest phase runs from 1-10, with just 0.36/0.01 mm/in. [...] In Haldwani, March brings rainfall levels ranging from 0.36/0.01 mm/in to 2.64/0.10 mm/in.\\n13.03 is 

In [72]:
zarvis_2_o.graph.get_state(thread).next

()

In [73]:
#just a custom input box for human input
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in zarvis_2_o.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
while zarvis_2_o.graph.get_state(thread).next:
    print("\n", zarvis_2_o.graph.get_state(thread),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in zarvis_2_o.graph.stream(None, thread):
        for v in event.values():
            print(v)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='b5b0e8a9-2ae5-4ca4-8da3-2510f3750df1'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'cw7xr03yf', 'function': {'arguments': '{"query":"LA weather today"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 349, 'total_tokens': 369, 'completion_time': 0.033447152, 'completion_tokens_details': None, 'prompt_time': 0.027882375, 'prompt_tokens_details': None, 'queue_time': 0.005842795, 'total_time': 0.061329527}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-94b1-7792-89d8-08ff62416845-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'LA weather today'}, 'id': 'cw7xr03yf', 'type': 'tool_call'}], in

### RUN until the interrupt and then modify the state

In [74]:
messages = [HumanMessage("Whats the weather in Bageshwar?")]
thread = {"configurable": {"thread_id": "3"}}
for event in zarvis_2_o.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [HumanMessage(content='Whats the weather in Bageshwar?', additional_kwargs={}, response_metadata={}, id='596d3dc8-8acc-4be7-bd25-9e56bffb5b54'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3pkpv5tnv', 'function': {'arguments': '{"query":"Bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.03720304, 'completion_tokens_details': None, 'prompt_time': 0.032649566, 'prompt_tokens_details': None, 'queue_time': 0.00539226, 'total_time': 0.069852606}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-b2d4-7563-ae2e-2b70add0142f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Bageshwar weather'}, 'id': '3pkpv5tnv', 'type': 'tool_call

In [75]:
zarvis_2_o.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in Bageshwar?', additional_kwargs={}, response_metadata={}, id='596d3dc8-8acc-4be7-bd25-9e56bffb5b54'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3pkpv5tnv', 'function': {'arguments': '{"query":"Bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.03720304, 'completion_tokens_details': None, 'prompt_time': 0.032649566, 'prompt_tokens_details': None, 'queue_time': 0.00539226, 'total_time': 0.069852606}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-b2d4-7563-ae2e-2b70add0142f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Bageshwar weather'}, 'id': '3pkpv5tnv

In [76]:
current_values = zarvis_2_o.graph.get_state(thread)
current_values.values['messages'][-1]

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3pkpv5tnv', 'function': {'arguments': '{"query":"Bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.03720304, 'completion_tokens_details': None, 'prompt_time': 0.032649566, 'prompt_tokens_details': None, 'queue_time': 0.00539226, 'total_time': 0.069852606}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-b2d4-7563-ae2e-2b70add0142f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Bageshwar weather'}, 'id': '3pkpv5tnv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 352, 'output_tokens': 22, 'total_tokens': 374})

In [77]:
current_values.values['messages'][-1].tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': 'Bageshwar weather'},
  'id': '3pkpv5tnv',
  'type': 'tool_call'}]

In [78]:
# we will update the state, where we will change the query in tool_calls step where we change the argument
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in Nainital'},
  'id': _id}
]

In [79]:
zarvis_2_o.graph.update_state(thread, current_values.values)

{'messages': [HumanMessage(content='Whats the weather in Bageshwar?', additional_kwargs={}, response_metadata={}, id='596d3dc8-8acc-4be7-bd25-9e56bffb5b54'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3pkpv5tnv', 'function': {'arguments': '{"query":"Bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.03720304, 'completion_tokens_details': None, 'prompt_time': 0.032649566, 'prompt_tokens_details': None, 'queue_time': 0.00539226, 'total_time': 0.069852606}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-b2d4-7563-ae2e-2b70add0142f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Nainital'}, 'id': '3pkpv5tnv'}], invali

{'configurable': {'thread_id': '3',
  'checkpoint_ns': '',
  'checkpoint_id': '1f11ebb3-0c07-65e1-8002-b54d5ede8326'}}

In [80]:
for event in zarvis_2_o.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Nainital'}, 'id': '3pkpv5tnv', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Weather Nainital in March 2026: Temperature & Climate\', \'url\': \'https://en.climate-data.org/asia/india/uttarakhand/nainital-33829/t/march-3/\', \'content\': \'| 9. March | 13 °C | 55 °F | 19 °C | 66 °F | 6 °C | 43 °F | 0.2 mm | 0.0 inch. |\\n| 10. March | 12 °C | 54 °F | 19 °C | 66 °F | 6 °C | 43 °F | 1.0 mm | 0.0 inch. |\\n| 11. March | 13 °C | 55 °F | 19 °C | 66 °F | 6 °C | 44 °F | 0.5 mm | 0.0 inch. |\\n| 12. March | 13 °C | 56 °F | 19 °C | 67 °F | 7 °C | 44 °F | 0.2 mm | 0.0 inch. |\\n| 13. March | 13 °C | 56 °F | 20 °C | 67 °F | 7 °C | 45 °F | 0.4 mm | 0.0 inch. |\\n| 14. March | 13 °C | 55 °F | 19 °C | 66 °F | 7 °C | 45 °F | 0.3 mm | 0.0 inch. |\\n| 15. March | 13 °C | 56 °F | 19 °C | 67 °F | 7 °C | 45 °F | 0.6 mm | 0.0 inch. |\\n| 16. March | 14 °C | 56 °F | 20 °C | 68 °F

### We will go back to state history and modify something there

In [81]:
states = []
for state in zarvis_2_o.graph.get_state_history(thread):
    print(state)
    print('--')
    states.append(state)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in Bageshwar?', additional_kwargs={}, response_metadata={}, id='596d3dc8-8acc-4be7-bd25-9e56bffb5b54'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3pkpv5tnv', 'function': {'arguments': '{"query":"Bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.03720304, 'completion_tokens_details': None, 'prompt_time': 0.032649566, 'prompt_tokens_details': None, 'queue_time': 0.00539226, 'total_time': 0.069852606}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-b2d4-7563-ae2e-2b70add0142f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Nainital'}, 'id': 

In [82]:
to_replay = states[-3]
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in Bageshwar?', additional_kwargs={}, response_metadata={}, id='596d3dc8-8acc-4be7-bd25-9e56bffb5b54'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3pkpv5tnv', 'function': {'arguments': '{"query":"Bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.03720304, 'completion_tokens_details': None, 'prompt_time': 0.032649566, 'prompt_tokens_details': None, 'queue_time': 0.00539226, 'total_time': 0.069852606}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-b2d4-7563-ae2e-2b70add0142f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Bageshwar weather'}, 'id': '3pkpv5tnv

In [83]:
for event in zarvis_2_o.graph.stream(None, to_replay.config):
    for k, v in event.items():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'Bageshwar weather'}, 'id': '3pkpv5tnv', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Bageshwar, Uttarakhand, India Weather Forecast 2026\', \'url\': \'https://www.tripclap.com/places/bageshwar/weather\', \'content\': "Understanding the weather patterns in Bageshwar is essential for making the most of your travel experience. Our detailed forecast includes temperature highs and lows, humidity levels, wind speed and direction, and current weather conditions to help you prepare for your journey.\\n\\n### 7-Day Weather Forecast\\n\\nBageshwar, Uttarakhand\\n\\nOvercast clouds\\nOvercast clouds\\nOvercast clouds\\nOvercast clouds\\nOvercast clouds\\nOvercast clouds\\nLight rain\\n\\nLast updated: Mar 13, 2026 2:24 PM\\n\\nWeather data provided by OpenWeatherMap\\n\\n### Understanding the Weather Forecast\\n\\nOur weather forecast for Bageshwar provides detailed information about ex

In [84]:
# we will go back in time and edit this

In [85]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in Bageshwar?', additional_kwargs={}, response_metadata={}, id='596d3dc8-8acc-4be7-bd25-9e56bffb5b54'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3pkpv5tnv', 'function': {'arguments': '{"query":"Bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.03720304, 'completion_tokens_details': None, 'prompt_time': 0.032649566, 'prompt_tokens_details': None, 'queue_time': 0.00539226, 'total_time': 0.069852606}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-b2d4-7563-ae2e-2b70add0142f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Bageshwar weather'}, 'id': '3pkpv5tnv

In [86]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in Bageshwar, accuweather'},
  'id': _id}]

In [87]:
branch_state = zarvis_2_o.graph.update_state(to_replay.config, to_replay.values)

{'messages': [HumanMessage(content='Whats the weather in Bageshwar?', additional_kwargs={}, response_metadata={}, id='596d3dc8-8acc-4be7-bd25-9e56bffb5b54'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3pkpv5tnv', 'function': {'arguments': '{"query":"Bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.03720304, 'completion_tokens_details': None, 'prompt_time': 0.032649566, 'prompt_tokens_details': None, 'queue_time': 0.00539226, 'total_time': 0.069852606}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-b2d4-7563-ae2e-2b70add0142f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Bageshwar, accuweather'}, 'id': '3pkpv5

In [88]:
for event in zarvis_2_o.graph.stream(None, branch_state):
    for k, v in event.items():
        if k != "__end__":
            print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Bageshwar, accuweather'}, 'id': '3pkpv5tnv', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Bageshwar, Uttarakhand, India Weather Forecast - AccuWeather\', \'url\': \'https://www.accuweather.com/en/in/bageshwar/201524/weather-forecast/201524\', \'content\': "# Bageshwar, Uttarakhand\\n\\nBageshwar\\n\\nUttarakhand\\n\\n## Around the Globe\\n\\nAround the Globe\\n\\n### Hurricane Tracker\\n\\n### Severe Weather\\n\\n### Radar & Maps\\n\\n### News & Features\\n\\n### Astronomy\\n\\n### Business\\n\\n### Climate\\n\\n### Health\\n\\n### Recreation\\n\\n### Sports\\n\\n### Travel\\n\\n### Warnings\\n\\n### Data Suite\\n\\n### Forensics\\n\\n### Advertising\\n\\n### Superior Accuracy™\\n\\n### Video\\n\\n## Today\\n\\n## Today\'s Weather\\n\\nFri, Mar 13\\n\\nTurning cloudy\\nHi: 83°\\n\\nTonight: Partly cloudy\\nLo: 50°\\n\\n## Current Weather\\n\\n2:31 PM\\n\\n#

## when we do not want to call tavily search tool and just want to return the answer without calling tool

In [89]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']

In [90]:
state_update = {"messages": [ToolMessage(
    tool_call_id=_id,
    name="tavily_search_results_json",
    content="54 degree celcius",
)]}

In [91]:
branch_and_add = zarvis_2_o.graph.update_state(
    to_replay.config,
    state_update,
    as_node="action")

In [92]:
for event in zarvis_2_o.graph.stream(None, branch_and_add):
    for k, v in event.items():
        print(v)

{'messages': [HumanMessage(content='Whats the weather in Bageshwar?', additional_kwargs={}, response_metadata={}, id='596d3dc8-8acc-4be7-bd25-9e56bffb5b54'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3pkpv5tnv', 'function': {'arguments': '{"query":"Bageshwar weather"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 352, 'total_tokens': 374, 'completion_time': 0.03720304, 'completion_tokens_details': None, 'prompt_time': 0.032649566, 'prompt_tokens_details': None, 'queue_time': 0.00539226, 'total_time': 0.069852606}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce66d-b2d4-7563-ae2e-2b70add0142f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Bageshwar weather'}, 'id': '3pkpv5tnv', 'type': 'tool_call